Import Dependencies

In [41]:
import openai
from qdrant_client import QdrantClient
from langsmith import Client

from openai import AsyncOpenAI

from ragas.llms import llm_factory
from ragas.embeddings.base import embedding_factory
# from ragas.embeddings import OpenAIEmbeddings

from ragas.dataset_schema import SingleTurnSample

from ragas.metrics.collections import Faithfulness, AnswerRelevancy

from ragas.metrics import IDBasedContextPrecision, IDBasedContextRecall


/var/folders/k8/vrzqp0495l91h236b45rqk7h0000gn/T/ipykernel_81268/220255632.py:15: DeprecationWarning: Importing IDBasedContextPrecision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import IDBasedContextPrecision
  from ragas.metrics import IDBasedContextPrecision, IDBasedContextRecall
/var/folders/k8/vrzqp0495l91h236b45rqk7h0000gn/T/ipykernel_81268/220255632.py:15: DeprecationWarning: Importing IDBasedContextRecall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import IDBasedContextRecall
  from ragas.metrics import IDBasedContextPrecision, IDBasedContextRecall


Download an example reference data point from LangSmith

In [42]:
from dotenv import load_dotenv

load_dotenv("../../.env")

False

In [43]:
ls_client = Client()

In [44]:
dataset = ls_client.read_dataset(
    dataset_name="rag-evaluation-dataset"
)

In [45]:
dataset

Dataset(name='rag-evaluation-dataset', description='RAG evaluation dataset', data_type=<DataType.kv: 'kv'>, id=UUID('51ed3170-73b3-4d73-abbd-58c7bdd16e29'), created_at=datetime.datetime(2026, 10, 25, 5, 32, 44, 814373, tzinfo=TzInfo(0)), modified_at=datetime.datetime(2026, 10, 25, 5, 32, 44, 814373, tzinfo=TzInfo(0)), example_count=60, session_count=0, last_session_start_time=None, inputs_schema=None, outputs_schema=None, transformations=None, metadata={'runtime': {'sdk': 'langsmith-py', 'library': 'langsmith', 'runtime': 'python', 'platform': 'macOS-15.6-arm64-arm-64bit', 'sdk_version': '0.14.12', 'runtime_version': '3.12.14', 'langchain_version': None, 'py_implementation': 'CPython', 'langchain_core_version': None}})

In [46]:
list(ls_client.list_examples(dataset_id=dataset.id, limit=50))[25].outputs

{'ground_truth': 'For cooling laptops, there is the AICHESON laptop cooler pad with 6 fans, adjustable fan speeds, ergonomic height settings, and compatibility with 13 to 15.6 inch laptops. For routers, TV boxes, receivers, modems and similar electronics, the Marame 120mm USB fan provides adjustable airflow and dust filters. For internal CPU/GPU or console cooling maintenance, ARCTIC MX-6 thermal paste offers high thermal conductivity, long durability, and non-conductive application.',
 'reference_context_ids': ['B099N9F3FP', 'B0BRJS644Z', 'B09VDLH5M6'],
 'reference_descriptions': ['AICHESON Laptop Cooler Pad with 6 Cooling Fans for 11" - 15.6" Laptops, RED Lights Super Laptop Cooling Fans - AICHESON cooling pads laptop is specially designed with 6 (70mm) cooling fans, each fan speed is 1500-2200RPM from level 1 to level 6, providing you adjustable airflows to lengthen the life of your laptop Ergonomic Heights Stands - Once the hook is dropped, it is locked in position, and it is very 

In [47]:
list(ls_client.list_examples(dataset_id=dataset.id, limit=50))[25].inputs

{'question': 'What products do you sell that can help keep laptops or electronics cooler?'}

In [48]:
reference_input = list(ls_client.list_examples(dataset_id=dataset.id, limit=50))[25].inputs
reference_output = list(ls_client.list_examples(dataset_id=dataset.id, limit=50))[25].outputs

RAG Pipeline

In [49]:
qdrant_client = QdrantClient("url=http://localhost:6333")

def get_embedding(text, model="text-embedding-3-small"):
    response = openai.embeddings.create(
        input=text,
        model=model
    )

    return response.data[0].embedding


def retrieve_data(query, qdrant_client, k=5):

    query_embedding = get_embedding(query)

    results = qdrant_client.query_points(
        collection_name="Amazon-items-collection-01",
        query=query_embedding,
        limit=k
    )

    retrieved_context_ids = []
    retrieved_context = []
    similarity_scores = []
    retrieved_context_ratings = []

    for result in results.points:
        retrieved_context_ids.append(result.payload["parent_asin"])
        retrieved_context.append(result.payload["preprocessed_description"])
        similarity_scores.append(result.score)
        retrieved_context_ratings.append(result.payload["average_rating"])

    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_context": retrieved_context,
        "similarity_scores": similarity_scores,
        "retrieved_context_ratings": retrieved_context_ratings
    }


def process_context(context):

    formatted_context = ""

    for id, chunk, rating in zip(context["retrieved_context_ids"], context["retrieved_context"], context["retrieved_context_ratings"]):
        formatted_context += f"- ID: {id}, rating: {rating}, description: {chunk}\n"

    return formatted_context


def build_prompt(preprocessed_context, query):

    prompt = f"""
You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructions:
- Answer the question based on the provided context only.
- Never use word context and refer to it as the available products.
- Do not use markdown formatting.

Context:
{preprocessed_context}

Question:
{query}    
"""

    return prompt


def generate_answer(prompt):

    response = openai.chat.completions.create(
        model="gpt-5.4-nano",
        messages=[
            {"role": "system", "content": prompt}
        ],
        reasoning_effort="none"
    )

    return response.choices[0].message.content


def rag_pipeline(query, qdrant_client, top_k=5):

    retrieved_context = retrieve_data(query, qdrant_client, top_k)
    preprocessed_context = process_context(retrieved_context)
    prompt = build_prompt(preprocessed_context, query)
    answer = generate_answer(prompt)

    final_answer = {
        "answer": answer,
        "question": query,
        "retrieved_context_ids": retrieved_context["retrieved_context_ids"],
        "retrieved_context": retrieved_context["retrieved_context"]
    }

    return final_answer

In [50]:
rag_pipeline("Can I get a charger?", qdrant_client)

{'answer': 'Yes. We have a few chargers/cables available:\n\n1) iPhone Charger (Lightning) – 6ft 3-pack, MFi certified, fast charging up to 3A and supports data sync (compatible with many iPhone and iPad models).\n\n2) Notebook Charger (White, 60W) – replacement charger for notebook computers (please confirm your specific Mac notebook model before buying).\n\n3) USB-C to USB-C Cable – 6.6ft 100W PD (5A), for charging USB-C devices like phones, tablets, and many laptops (includes a 3-year warranty).\n\n4) iPhone Charger Cord – MFi certified Lightning cable (1m/3.3ft), fast charging and high-speed data sync (comes as a single cable).',
 'question': 'Can I get a charger?',
 'retrieved_context_ids': ['B0BBVJJRHD',
  'B0BGH3H1WM',
  'B0BN1CMWCP',
  'B0C9QZS95R',
  'B0BXC72RLD'],
 'retrieved_context': ['iPhone Charger 6ft 3Pack, Apple MFi Certified Lightning Cable Fast Charging Long iPhone Charger Cord High Speed Data Sync Cable Compatible iPhone 14 13 12 11 Pro Max XS XR X 8 7 6S 6 Plus SE 

RAGAS Metrics

In [51]:
reference_input

{'question': 'What products do you sell that can help keep laptops or electronics cooler?'}

In [52]:
reference_output

{'ground_truth': 'For cooling laptops, there is the AICHESON laptop cooler pad with 6 fans, adjustable fan speeds, ergonomic height settings, and compatibility with 13 to 15.6 inch laptops. For routers, TV boxes, receivers, modems and similar electronics, the Marame 120mm USB fan provides adjustable airflow and dust filters. For internal CPU/GPU or console cooling maintenance, ARCTIC MX-6 thermal paste offers high thermal conductivity, long durability, and non-conductive application.',
 'reference_context_ids': ['B099N9F3FP', 'B0BRJS644Z', 'B09VDLH5M6'],
 'reference_descriptions': ['AICHESON Laptop Cooler Pad with 6 Cooling Fans for 11" - 15.6" Laptops, RED Lights Super Laptop Cooling Fans - AICHESON cooling pads laptop is specially designed with 6 (70mm) cooling fans, each fan speed is 1500-2200RPM from level 1 to level 6, providing you adjustable airflows to lengthen the life of your laptop Ergonomic Heights Stands - Once the hook is dropped, it is locked in position, and it is very 

In [53]:
result = rag_pipeline(reference_input["question"], qdrant_client)

In [54]:
result

{'answer': 'Here are the available products that can help keep laptops or electronics cooler:\n\n1) AICHESON Laptop Cooler Pad with 6 Cooling Fans (11"–15.6" laptops) (ID: B099N9F3FP)\n- 6 built-in 70mm fans with adjustable fan speed (1500–2200 RPM, levels 1–6)\n- Ergonomic adjustable stand angles (10–27 degrees)\n- Low noise level (25–28 dBA)\n- LCD control panel to adjust fan speed and extra USB ports for peripherals\n\n2) Marame 120mm USB Powered Fan with Speed Controller (for electronics like router/modem/Xbox/TV boxes) (ID: B0BRJS644Z)\n- 120mm x 120mm USB-powered fan with speed control (off/low/medium/high)\n- Works on many USB power sources (power bank, adapters, car chargers, etc.)\n- Includes rubber feet and a dust filter\n\n3) ARCTIC MX-6 Thermal Paste (CPU/GPU/laptops/consoles) (ID: B09VDLH5M6)\n- High-performance, non-conductive, non-capacitive thermal paste to improve heat transfer\n\n4) ZARIMI Compressed Air Duster (cordless electric) (ID: B0C8DBH7ZT)\n- Powerful electric

In [55]:
openai_client = AsyncOpenAI()

In [56]:
ragas_llm = llm_factory("gpt-5.4-mini", client=openai_client, max_completion_tokens=4000)

args = ragas_llm.model_args
args.pop("max_tokens", None)
args.pop("top_p", None)
args["temperature"] = 0.0
args["max_completion_tokens"] = 4000

ragas_embeddings = embedding_factory("openai", model="text-embedding-3-small", client=openai_client)

In [57]:
async def ragas_faitfulness(run, example):

    scorer = Faithfulness(llm=ragas_llm)

    return await scorer.ascore(
        user_input=run["question"],
        response=run["answer"],
        retrieved_contexts=run["retrieved_context"]
    )

In [58]:
await ragas_faitfulness(result, reference_output)

MetricResult(value=0.8)

In [59]:
async def ragas_relevancy(run, example):

    scorer = AnswerRelevancy(llm=ragas_llm, embeddings=ragas_embeddings)

    return await scorer.ascore(
        user_input=run["question"],
        response=run["answer"]
    )

In [60]:
await ragas_relevancy(result, reference_output)

MetricResult(value=0.8749996171756065)

In [61]:
async def ragas_context_precision_id_based(run, example):

    sample = SingleTurnSample(
        retrieved_context_ids=run["retrieved_context_ids"],
        reference_context_ids=example["reference_context_ids"]
    )

    scorer = IDBasedContextPrecision()

    return await scorer.single_turn_ascore(sample)

In [62]:
await ragas_context_precision_id_based(result, reference_output)

0.6

In [63]:
async def ragas_context_recall_id_based(run, example):

    sample = SingleTurnSample(
        retrieved_context_ids=run["retrieved_context_ids"],
        reference_context_ids=example["reference_context_ids"]
    )

    scorer = IDBasedContextRecall()

    return await scorer.single_turn_ascore(sample)

In [64]:
await ragas_context_recall_id_based(result, reference_output)

1.0